In [15]:
from pyspark.sql import SparkSession
import configparser
import os

config = configparser.ConfigParser()
config.read(os.path.join(os.path.dirname(os.path.abspath("__file__")), 'conf/config.ini'))
url = 'jdbc:mysql://localhost/{}'.format(config.get('mysql', 'database'))
driver = config.get('mysql', 'driver')
tablename = "Wines"
username = config.get('mysql', 'username')
password = config.get('mysql', 'password')
print("url : ", url)
print("Driver : ", driver)
print("MySQL User : ", username)

url :  jdbc:mysql://localhost/stocksdb
Driver :  com.mysql.cj.jdbc.Driver
MySQL User :  quizadmin


In [16]:
spark = SparkSession.builder.appName("Wine Prediction") \
.master("local[1]") \
.config("spark.jars",
                "/Users/gaurav/.m2/repository/com/mysql/mysql-connector-j/8.0.33/mysql-connector-j-8.0.33.jar") \
.getOrCreate()

In [17]:
wine_df = spark.read.format("jdbc").options(url=url, driver=driver, user=username, password=password, dbtable=tablename).load()
#spark.sparkContext.stop()

In [18]:
wine_df.show(10)

+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+------+
|fixed_acidity|volatile_acidity|citric_acid|residual_sugar|chlorides|free_sulfur_dioxide|total_sulfur_dioxide|density|  ph|sulphates|alcohol|quality|is_red|
+-------------+----------------+-----------+--------------+---------+-------------------+--------------------+-------+----+---------+-------+-------+------+
|          7.4|             0.7|        0.0|           1.9|    0.076|               11.0|                34.0| 0.9978|3.51|     0.56|    9.4|      5|     1|
|          7.8|            0.88|        0.0|           2.6|    0.098|               25.0|                67.0| 0.9968| 3.2|     0.68|    9.8|      5|     1|
|          7.8|            0.76|       0.04|           2.3|    0.092|               15.0|                54.0|  0.997|3.26|     0.65|    9.8|      5|     1|
|         11.2|            0.28|       0.56|           1.9

In [19]:
from pyspark.ml.feature import VectorAssembler

## VectorAssembler

VectorAssembler is a feature transformer in Apache Spark's Machine Learning (ML) library, specifically in the pyspark.ml.feature module. It is used for assembling multiple columns of data into a single vector column, which is a common requirement when working with machine learning algorithms in Spark.

The primary purpose of VectorAssembler is to prepare data for machine learning pipelines by combining various features into a single feature vector. This is important because many machine learning algorithms in Spark expect input data in the form of a single vector column, where each element of the vector corresponds to a feature.

##what is the functionality of seed in randomSplit function

In Apache Spark's randomSplit function, the seed parameter is used to specify a random seed for generating random numbers when splitting a dataset into multiple subsets. The randomSplit function is often used for creating training and testing datasets or for other purposes where you need to divide your data randomly into two or more parts.

Here's how it works:

When you use randomSplit without specifying a seed, it generates a random split every time you call the function. This means that if you call randomSplit multiple times without a seed, you might get different random splits each time, even if the input data remains the same. This behavior is suitable when you want different random subsets for experimentation or testing.
When you specify a seed value, you're essentially setting the initial state for the random number generator. This ensures that, given the same input data and the same seed value, you will always get the same random split. It provides reproducibility, which can be important when you want to compare results or share your code with others.

In [20]:
train_df, test_df = wine_df.randomSplit([.8, .2], seed=12345)
predictors = ["fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar", "chlorides",
              "free_sulfur_dioxide", "total_sulfur_dioxide", "density", "ph", "sulphates", "alcohol"]
vec_assembler = VectorAssembler(inputCols=predictors, outputCol="features")
vec_train_df = vec_assembler.transform(train_df)
vec_train_df.select("features", "is_red").show(5)

+--------------------+------+
|            features|is_red|
+--------------------+------+
|[3.8,0.31,0.02,11...|     0|
|[3.9,0.225,0.4,4....|     0|
|[4.2,0.17,0.36,1....|     0|
|[4.2,0.215,0.23,5...|     0|
|[4.4,0.32,0.39,4....|     0|
+--------------------+------+
only showing top 5 rows



In [21]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(labelCol="is_red", featuresCol="features")
lr_model = lr.fit(vec_train_df)
vec_test_df = vec_assembler.transform(test_df)
predictions = lr_model.transform(vec_test_df)

23/09/10 17:13:04 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
23/09/10 17:13:04 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


In [22]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[vec_assembler, lr])
pipeline_model = pipeline.fit(train_df)
predictions = pipeline_model.transform(test_df)

In [23]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="is_red")
evaluator.evaluate(predictions)

0.9915751842624236